# ⚡ Vid2Rig — Free Kaggle GPU + Cloudflare Tunnel

Turn any video into 3D motion capture on rigged Mixamo characters with **MediaPipe + Three.js + optional GVHMR GPU solver**.

### ⚙️ Kaggle Notebook Settings:
1. **Accelerator**: Select **GPU T4 x2** or **GPU P100** (Free 30 hrs/week)
2. **Internet**: Toggle **Internet ON** in the right-hand panel (Settings → Internet → ON)
3. Run the cells below in order!

### Step 1: Verify GPU Environment

In [ ]:
!nvidia-smi

### Step 2: Install System Dependencies & Cloudflare Tunnel
Installs Node.js, ffmpeg, and downloads the official `cloudflared` Linux binary.

In [ ]:
!apt-get update -qq && apt-get install -y -qq nodejs npm ffmpeg
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
!cloudflared --version

### Step 3: Setup Vid2Rig Repository
If not already running inside the cloned repo, this clones it and installs npm dependencies.

In [ ]:
import os

if not os.path.exists("server.mjs"):
    !git clone https://github.com/liladhiee-web/vid2rig.git
    %cd vid2rig

!npm install --omit=dev
!mkdir -p uploads takes

### Step 4: Launch Web Server & Cloudflare Quick Tunnel
This starts the zero-dependency web server and establishes a secure Cloudflare HTTPS tunnel.
Click the generated **`https://*.trycloudflare.com`** URL to open Vid2Rig!

In [ ]:
import subprocess
import time
import re
from IPython.display import HTML, display

# 1. Start Node.js Web Server in background
server_log = open("/tmp/vid2rig_server.log", "w")
server_proc = subprocess.Popen(["node", "server.mjs"], stdout=server_log, stderr=subprocess.STDOUT)
time.sleep(2)

# 2. Start Cloudflare Tunnel
tunnel_cmd = ["cloudflared", "tunnel", "--url", "http://127.0.0.1:5177", "--no-autoupdate"]
tunnel_proc = subprocess.Popen(tunnel_cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)

url_pattern = re.compile(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com")
tunnel_url = None

print("Starting Cloudflare Tunnel, searching for public URL...")
for line in tunnel_proc.stdout:
    match = url_pattern.search(line)
    if match:
        tunnel_url = match.group(0)
        break

if tunnel_url:
    display(HTML(f"""
    <div style="background:#0a0f0d;border:2px solid #00ff88;padding:20px;border-radius:8px;margin:15px 0;box-shadow:0 0 25px rgba(0,255,136,0.35);font-family:sans-serif;">
        <h2 style="color:#00ff88;margin:0 0 8px 0;letter-spacing:0.5px;">⚡ Vid2Rig Studio is Live!</h2>
        <p style="color:#e6f7ef;margin:0 0 16px 0;font-size:14px;">Your Kaggle GPU session is now connected via Cloudflare Tunnel:</p>
        <a href="{tunnel_url}" target="_blank" style="background:#00ff88;color:#000;font-weight:bold;font-size:15px;padding:12px 24px;text-decoration:none;border-radius:4px;display:inline-block;letter-spacing:0.5px;">
            🚀 OPEN VID2RIG STUDIO &rarr;
        </a>
        <p style="color:#769a87;font-size:12px;margin:14px 0 0 0;font-family:monospace;">Direct Link: <a href="{tunnel_url}" target="_blank" style="color:#00ff88;">{tunnel_url}</a></p>
    </div>
    """))
    print(f"\n>>> Public URL: {tunnel_url}\n")
else:
    print("Cloudflare tunnel failed to produce a public URL. Check logs:")

# Keep running
try:
    for line in tunnel_proc.stdout:
        pass
except KeyboardInterrupt:
    print("Stopping server and tunnel...")
    server_proc.terminate()
    tunnel_proc.terminate()


### Step 5: (Optional) Run the Python HQ Worker (GVHMR + WiLoR)
Run this in a separate notebook or background task if you want the high-fidelity GVHMR motion solver running on the Kaggle GPU.

In [ ]:
# Install HQ Python dependencies
# !pip install -r hq/requirements.txt
# !python hq/worker.py
print("HQ Worker can be run if SMPL-X body models are provided.")
